In [0]:
import base64
from pyspark.sql import functions as F

# Configuration
CATALOG = "main"
SCHEMA = "default"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/my_images/sample_image.jpg"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.image_embeddings"
MODEL_ENDPOINT = "databricks-nomic-embed-vision-v1" # Example native endpoint

# 1. Read the JPG from Volume
with open(VOLUME_PATH, "rb") as f:
    img_binary = f.read()
    img_b64 = base64.b64encode(img_binary).decode("utf-8")

# 2. Create a DataFrame with the image data
# Genie Space works best with metadata, so we include the path and a description
data = [(VOLUME_PATH, img_b64, "Sample image description")]
df = spark.createDataFrame(data, ["file_path", "image_base64", "description"])

# 3. Generate Embeddings using Native AI Function
# Note: You can also do this directly in SQL using ai_query()
df_with_embeddings = df.withColumn(
    "embedding", 
    F.expr(f"ai_query('{MODEL_ENDPOINT}', named_struct('image', image_base64))")
)

# 4. Save to a Delta Table for Genie Space
# We drop the raw base64 to keep the table size manageable
df_with_embeddings.drop("image_base64").write.mode("overwrite").saveAsTable(TABLE_NAME)

print(f"Table saved successfully to {TABLE_NAME}")
